# Optical bench object tutorial

This notebook is a compact tutorial for the core geometric objects in `optics_simulator.beam_trace`.

It demonstrates:

- `Beam3D` / `Ray3D` throughput bookkeeping
- `RotatingPhaseScreen3D`
- `Lens3D`
- `Mirror3D`
- `AmplitudePhaseMask3D`
- `DeformableMirror3D`
- `ShackHartmannWFS3D`
- the standardised `OpticalBench3D.plot_3d(...)` bench visualisation
- amplitude and phase/OPD cross-sections sampled from each element
- several DM command shapes producing different phase aberrations

The plotting wrappers in this notebook are deliberately thin: they call module methods such as `sample_uv()`, `sample_amplitude_uv()`, `contains_uv()`, `lenslet_mask_for_grid()`, and `bench.plot_3d()` rather than re-implementing the optical elements.

In [ ]:
# Imports and repository setup

from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
for root in [cwd, cwd.parent, cwd.parent.parent]:
    if (root / "optics_simulator").exists():
        REPO_ROOT = root
        break
else:
    raise RuntimeError("Could not find repository root containing optics_simulator/")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from optics_simulator import beam_trace as bt

print("Repository root:", REPO_ROOT)
print("beam_trace:", bt.__file__)

required = [
    "Ray3D",
    "Beam3D",
    "OpticalBench3D",
    "RotatingPhaseScreen3D",
    "Lens3D",
    "Mirror3D",
    "AmplitudePhaseMask3D",
    "DeformableMirror3D",
    "ShackHartmannWFS3D",
]

missing = [name for name in required if not hasattr(bt, name)]
for name in required:
    print(f"{name:28s}", hasattr(bt, name))

if missing:
    raise RuntimeError("Missing required beam_trace classes: " + ", ".join(missing))

## Shared tutorial parameters and plotting wrappers

The helper functions below are only for visualisation. They sample each object using the object's own module methods:

- `sample_uv()` for OPD-producing elements
- `sample_amplitude_uv()` for amplitude masks
- `contains_uv()` for geometric apertures
- `lenslet_mask_for_grid()` and `lenslet_centres()` for the SH-WFS
- `OpticalBench3D.plot_3d()` for 3D geometry

In [ ]:
# Shared parameters

wavelength = 633e-9
beam_radius = 6.0e-3
beam_diameter = 2 * beam_radius
npix = 256
t_demo = 0.15

# A simple collimated beam travelling along +z.
base_beam = bt.Beam3D.collimated_circular(
    radius=beam_radius,
    nrings=3,
    nphi=12,
    origin=(0.0, 0.0, -60e-3),
    direction=(0.0, 0.0, 1.0),
    wavelength=wavelength,
    label="collimated beam",
)


def uv_grid(extent_m=16e-3, npix=256):
    u = np.linspace(-0.5 * extent_m, 0.5 * extent_m, npix)
    v = np.linspace(-0.5 * extent_m, 0.5 * extent_m, npix)
    uu, vv = np.meshgrid(u, v)
    uv = np.stack([uu, vv], axis=-1)
    return uu, vv, uv


def sample_object_response(elem, wavelength=wavelength, extent_m=16e-3, npix=256, t=0.0):
    """
    Visualisation-only sampler for a single object.

    It calls the element's module methods where available. For purely geometric
    ray elements such as Lens3D, the plot shows the clear aperture and the usual
    thin-lens equivalent OPD for tutorial context.
    """
    uu, vv, uv = uv_grid(extent_m=extent_m, npix=npix)

    # Default: full transmission, zero OPD.
    amp = np.ones_like(uu, dtype=float)
    valid_amp = np.ones_like(uu, dtype=bool)
    opd = np.zeros_like(uu, dtype=float)
    valid_opd = np.ones_like(uu, dtype=bool)

    # Amplitude transmission.
    if hasattr(elem, "sample_amplitude_uv"):
        amp, valid_amp = elem.sample_amplitude_uv(uv, t=t)
    elif hasattr(elem, "contains_uv"):
        valid_amp = elem.contains_uv(uv)
        amp = valid_amp.astype(float)
        if hasattr(elem, "field_reflectivity"):
            amp *= float(elem.field_reflectivity)
    elif hasattr(elem, "aperture_radius"):
        rr = np.sqrt(uu**2 + vv**2)
        valid_amp = rr <= float(elem.aperture_radius)
        amp = valid_amp.astype(float)

    # OPD / phase.
    if hasattr(elem, "sample_uv"):
        opd, valid_opd = elem.sample_uv(uv, t=t)
    elif type(elem).__name__ == "Lens3D":
        # Equivalent thin-lens OPD: phase = -k r^2/(2f), so OPD = -r^2/(2f).
        # Lens3D itself is geometric; this is shown for wave-optics intuition.
        rr2 = uu**2 + vv**2
        opd = -0.5 * rr2 / float(elem.f)
        valid_opd = amp > 0
    elif type(elem).__name__ == "Mirror3D":
        opd = np.full_like(uu, float(getattr(elem, "opd_offset_m", 0.0)))
        valid_opd = amp > 0
    elif type(elem).__name__ == "ShackHartmannWFS3D":
        # Build a lenslet aperture map and local thin-lens OPD per lenslet.
        amp = np.zeros_like(uu, dtype=float)
        opd = np.zeros_like(uu, dtype=float)
        valid = np.zeros_like(uu, dtype=bool)

        ny, nx = elem.n_lenslets
        centres_u, centres_v = elem.lenslet_centres()
        for iy in range(ny):
            for ix in range(nx):
                if elem.valid_lenslet_mask is not None and not elem.valid_lenslet_mask[iy, ix]:
                    continue
                m = elem.lenslet_mask_for_grid(uu, vv, (iy, ix))
                du = uu - centres_u[iy, ix]
                dv = vv - centres_v[iy, ix]
                opd_lenslet = -0.5 * (du**2 + dv**2) / float(elem.focal_length_m)
                amp[m] = 1.0
                opd[m] = opd_lenslet[m]
                valid[m] = True
        valid_amp = valid
        valid_opd = valid

    valid = valid_amp & valid_opd
    amp = np.where(valid, amp, 0.0)
    opd = np.where(valid, opd, np.nan)
    phase = 2.0 * np.pi * opd / wavelength

    return {
        "u": uu,
        "v": vv,
        "uv": uv,
        "amplitude": amp,
        "opd_m": opd,
        "phase_rad": phase,
        "valid": valid,
        "extent_mm": [uu.min() * 1e3, uu.max() * 1e3, vv.min() * 1e3, vv.max() * 1e3],
    }


def plot_object_response(elem, title=None, wavelength=wavelength, extent_m=16e-3, npix=256, t=0.0):
    resp = sample_object_response(elem, wavelength=wavelength, extent_m=extent_m, npix=npix, t=t)
    amp = resp["amplitude"]
    phase = resp["phase_rad"]
    opd_nm = resp["opd_m"] * 1e9
    extent = resp["extent_mm"]
    mid = npix // 2
    u_mm = resp["u"][mid, :] * 1e3

    title = elem.label if title is None else title

    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    ax = axes.ravel()

    im0 = ax[0].imshow(amp, origin="lower", extent=extent, vmin=0, vmax=max(1.0, np.nanmax(amp)))
    ax[0].set_title(f"{title}: field amplitude")
    ax[0].set_xlabel("u [mm]")
    ax[0].set_ylabel("v [mm]")
    ax[0].set_aspect("equal")
    plt.colorbar(im0, ax=ax[0], fraction=0.046, pad=0.04)

    finite_phase = np.isfinite(phase)
    if np.any(finite_phase):
        vmax_phase = np.nanpercentile(np.abs(phase[finite_phase]), 99)
        if not np.isfinite(vmax_phase) or vmax_phase <= 0:
            vmax_phase = 1.0
    else:
        vmax_phase = 1.0
    im1 = ax[1].imshow(phase, origin="lower", extent=extent, cmap="RdBu_r", vmin=-vmax_phase, vmax=vmax_phase)
    ax[1].set_title(f"{title}: phase [rad]")
    ax[1].set_xlabel("u [mm]")
    ax[1].set_ylabel("v [mm]")
    ax[1].set_aspect("equal")
    plt.colorbar(im1, ax=ax[1], fraction=0.046, pad=0.04)

    ax[2].plot(u_mm, amp[mid, :], lw=2)
    ax[2].set_title("central amplitude cross-section")
    ax[2].set_xlabel("u [mm]")
    ax[2].set_ylabel("field amplitude")
    ax[2].grid(True, alpha=0.3)

    ax[3].plot(u_mm, opd_nm[mid, :], lw=2)
    ax[3].set_title("central OPD cross-section")
    ax[3].set_xlabel("u [mm]")
    ax[3].set_ylabel("OPD [nm]")
    ax[3].grid(True, alpha=0.3)

    fig.tight_layout()
    return fig, resp


def plot_single_element_bench(elem, beam=base_beam, title=None, s_end=0.05, view=(24, -55)):
    bench = bt.OpticalBench3D()
    bench.add(elem)
    fig = bench.plot_3d(
        beams=[beam],
        t=t_demo,
        s_end=s_end,
        title=elem.label if title is None else title,
        max_rays_per_beam=80,
        draw_elements=True,
        draw_labels=True,
        draw_normals=True,
        draw_local_axes=True,
        equal_axes=True,
        orthographic=True,
        view=view,
    )
    return fig

## 1. Rotating phase screen

A `RotatingPhaseScreen3D` is an OPD-producing element. It exposes `sample_uv(uv, t)`, so it contributes directly to the pupil phase model.

In [ ]:
screen_map = bt.make_von_karman_opd_map(
    n=512,
    extent_m=30e-3,
    r0=0.035,
    L0=10.0,
    rms_opd_m=120e-9,
    seed=10,
)

screen = bt.RotatingPhaseScreen3D(
    point=[0.0, 0.0, -20e-3],
    normal=[0.0, 0.0, 1.0],
    opd_map=screen_map,
    map_extent_m=30e-3,
    clear_radius=14e-3,
    angular_velocity=2.0 * np.pi * 0.5,
    label="rotating phase screen",
)

plot_single_element_bench(screen, title="Rotating phase screen: geometric view", s_end=0.04)
fig, screen_resp = plot_object_response(screen, extent_m=18e-3, t=t_demo)
plt.show()

## 2. Geometric thin lens

`Lens3D` is a geometric paraxial thin lens. In the 3D ray plot it changes ray directions. For the amplitude/phase panel below, the aperture is shown and an equivalent thin-lens quadratic phase is plotted for wave-optics intuition. The `Lens3D` class itself remains a ray-optics element.

In [ ]:
lens = bt.Lens3D(
    point=[0.0, 0.0, 0.0],
    normal=[0.0, 0.0, 1.0],
    f=55e-3,
    aperture_radius=7.0e-3,
    label="thin lens",
)

plot_single_element_bench(lens, title="Lens3D: geometric ray kick", s_end=0.08)
fig, lens_resp = plot_object_response(lens, extent_m=16e-3)
plt.show()

## 3. Mirror

`Mirror3D` reflects rays geometrically and carries scalar field-amplitude reflectivity. The amplitude response here is just the clear aperture multiplied by the mirror's `field_reflectivity`.

In [ ]:
# Beam along +z reflected toward +x: choose n proportional to d_in - d_out.
mirror_normal = np.array([-1.0, 0.0, 1.0])
mirror_normal = mirror_normal / np.linalg.norm(mirror_normal)

mirror = bt.Mirror3D(
    point=[0.0, 0.0, 0.0],
    normal=mirror_normal,
    aperture_radius=10.0e-3,
    aperture_shape="circular",
    field_reflectivity=0.97,
    opd_offset_m=8e-9,
    label="fold mirror",
)

print("Input direction: ", base_beam.chief_ray.d)
print("Output direction:", mirror.reflect_direction(base_beam.chief_ray.d))

plot_single_element_bench(mirror, title="Mirror3D: reflected ray path", s_end=0.05, view=(25, -45))
fig, mirror_resp = plot_object_response(mirror, extent_m=22e-3)
plt.show()

## 4. Amplitude and phase mask / aperture

`AmplitudePhaseMask3D` exposes both `sample_amplitude_uv()` and `sample_uv()`. It can therefore contribute both field-amplitude transmission and OPD/phase delay to the sampled pupil field.

In [ ]:
map_n = 256
map_extent = 18e-3
u = np.linspace(-0.5 * map_extent, 0.5 * map_extent, map_n)
v = np.linspace(-0.5 * map_extent, 0.5 * map_extent, map_n)
uu, vv = np.meshgrid(u, v)
rr = np.sqrt(uu**2 + vv**2)

amp_map = 0.92 * np.exp(-0.15 * (rr / beam_radius) ** 2)
amp_map[rr > 5.3e-3] = 0.0

opd_mask = 45e-9 * np.sin(2 * np.pi * uu / map_extent) * np.cos(2 * np.pi * vv / map_extent)
opd_mask *= np.exp(-(rr / beam_radius) ** 4)

mask = bt.AmplitudePhaseMask3D(
    point=[0.0, 0.0, 0.0],
    normal=[0.0, 0.0, 1.0],
    aperture_shape="circular",
    aperture_radius=5.3e-3,
    amplitude=1.0,
    amplitude_map=amp_map,
    opd_map_m=opd_mask,
    map_extent_m=map_extent,
    ray_block_threshold=1e-6,
    label="amplitude + phase mask",
)

plot_single_element_bench(mask, title="AmplitudePhaseMask3D: aperture/throughput", s_end=0.04)
fig, mask_resp = plot_object_response(mask, extent_m=18e-3)
plt.show()

## 5. Deformable mirror: actuator influence functions and phase aberrations

The reflective `DeformableMirror3D` exposes a continuous actuator surface model. `sample_uv()` returns the reflective OPD map, while `surface_sag_uv()` returns the physical mirror surface sag. In this section we test several command shapes.

For the response plots, the DM is sampled as an OPD-producing element. For the 3D plot, the same object can also reflect rays when `reflect_rays=True`.

In [ ]:
act_shape = (12, 12)
ay = np.arange(act_shape[0]) - 0.5 * (act_shape[0] - 1)
ax = np.arange(act_shape[1]) - 0.5 * (act_shape[1] - 1)
AAx, AAy = np.meshgrid(ax, ay)
rho = np.sqrt(AAx**2 + AAy**2) / (0.5 * act_shape[0])
valid_act = rho <= 1.05

# Reflect incoming +z beam toward +x.
dm_normal = np.array([-1.0, 0.0, 1.0])
dm_normal = dm_normal / np.linalg.norm(dm_normal)

dm = bt.DeformableMirror3D(
    point=[0.0, 0.0, 0.0],
    normal=dm_normal,
    actuator_shape=act_shape,
    actuator_pitch_m=1.0e-3,
    surface_per_command_m=100e-9,
    commands=np.zeros(act_shape),
    valid_actuator_mask=valid_act,
    coupling=0.30,
    aperture_radius=10.0e-3,  # large enough for the projected reflective footprint
    aperture_shape="circular",
    reflect_rays=True,
    use_surface_normals=False,
    command_limits=(-1.0, 1.0),
    label="reflective DM",
)

print("Input direction: ", base_beam.chief_ray.d)
print("Output direction:", dm.reflect_direction(base_beam.chief_ray.d))

plot_single_element_bench(dm, title="Reflective DM: geometric reflection", s_end=0.05, view=(25, -45))
plt.show()

In [ ]:
# DM command patterns.
# These are not intended to be a complete modal basis; they are just sanity checks.

focus = (2.0 * rho**2 - 1.0) * valid_act
astig = (AAx**2 - AAy**2)
astig = astig / np.nanmax(np.abs(astig[valid_act])) * valid_act
coma_x = AAx * (3.0 * rho**2 - 2.0)
coma_x = coma_x / np.nanmax(np.abs(coma_x[valid_act])) * valid_act
single_poke = np.zeros(act_shape)
single_poke[act_shape[0] // 2, act_shape[1] // 2] = 1.0
waffle = ((-1.0) ** (AAx + AAy)) * valid_act

patterns = {
    "flat": np.zeros(act_shape),
    "focus-like": 0.20 * focus,
    "astigmatism-like": 0.20 * astig,
    "coma-like": 0.20 * coma_x,
    "single actuator poke": 0.35 * single_poke,
    "waffle-like": 0.08 * waffle,
}

fig, axes = plt.subplots(len(patterns), 3, figsize=(13, 3.0 * len(patterns)))

extent_m = 18e-3
uu, vv, uv = uv_grid(extent_m=extent_m, npix=npix)
mid = npix // 2
u_mm = uu[mid, :] * 1e3

for row, (name, cmd) in enumerate(patterns.items()):
    dm.set_commands(cmd)
    opd, valid = dm.sample_uv(uv, t=0.0)
    sag = dm.surface_sag_uv(uv)

    opd_plot = np.where(valid, opd * 1e9, np.nan)
    sag_plot = np.where(valid, sag * 1e9, np.nan)

    im0 = axes[row, 0].imshow(cmd, origin="lower")
    axes[row, 0].set_title(f"{name}: commands")
    plt.colorbar(im0, ax=axes[row, 0], fraction=0.046, pad=0.04)

    vmax = np.nanpercentile(np.abs(opd_plot), 99) if np.any(np.isfinite(opd_plot)) else 1.0
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0
    im1 = axes[row, 1].imshow(
        opd_plot,
        origin="lower",
        extent=[-0.5*extent_m*1e3, 0.5*extent_m*1e3, -0.5*extent_m*1e3, 0.5*extent_m*1e3],
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
    )
    axes[row, 1].set_title("sampled reflective OPD [nm]")
    axes[row, 1].set_xlabel("u [mm]")
    axes[row, 1].set_ylabel("v [mm]")
    axes[row, 1].set_aspect("equal")
    plt.colorbar(im1, ax=axes[row, 1], fraction=0.046, pad=0.04)

    axes[row, 2].plot(u_mm, opd_plot[mid, :], label="OPD [nm]")
    axes[row, 2].plot(u_mm, sag_plot[mid, :], label="surface sag [nm]", ls="--")
    axes[row, 2].set_title("central cross-section")
    axes[row, 2].set_xlabel("u [mm]")
    axes[row, 2].grid(True, alpha=0.3)
    axes[row, 2].legend()

plt.tight_layout()
plt.show()

# Restore a useful non-flat command for following cells.
dm.set_commands(patterns["astigmatism-like"])

## 6. Shack–Hartmann WFS lenslet array

`ShackHartmannWFS3D` is a compound lenslet-array element. It provides lenslet geometry, masks, detector-cell bookkeeping, and geometric microlens ray kicks. The plot below shows the lenslet apertures and the equivalent local quadratic lenslet phases for tutorial context.

In [ ]:
sh = bt.ShackHartmannWFS3D(
    point=[0.0, 0.0, 0.0],
    normal=[0.0, 0.0, 1.0],
    n_lenslets=(6, 6),
    lenslet_pitch_m=2.0e-3,
    lenslet_diameter_m=1.75e-3,
    lenslet_shape="square",
    focal_length_m=25e-3,
    wavelength_m=wavelength,
    detector_pixel_pitch_m=5.0e-6,
    pixels_per_lenslet=32,
    terminate_rays=False,
    label="SH-WFS lenslet array",
)

plot_single_element_bench(sh, title="ShackHartmannWFS3D: lenslet-array geometry", s_end=0.05)
fig, sh_resp = plot_object_response(sh, extent_m=14e-3)
plt.show()

print("Detector shape:", sh.detector_shape)
print("First lenslet detector cell bounds:", sh.detector_cell_bounds((0, 0)))

## 7. Small complete bench using all objects

This final cell chains several objects into a small folded bench and uses the standard `bench.plot_3d(...)` method. It is intentionally minimal: the physical propagation/SH spot generation belongs in the dedicated SH-WFS notebooks/examples.

In [ ]:
# Build a compact folded bench with all major element types.

beam = bt.Beam3D.collimated_circular(
    radius=beam_radius,
    nrings=3,
    nphi=12,
    origin=[0.0, 0.0, -80e-3],
    direction=[0.0, 0.0, 1.0],
    wavelength=wavelength,
    label="test beam",
)

screen.point = np.array([0.0, 0.0, -35e-3])

fold = bt.Mirror3D(
    point=[0.0, 0.0, 0.0],
    normal=mirror_normal,
    aperture_radius=12e-3,
    aperture_shape="circular",
    field_reflectivity=1.0,
    label="fold mirror",
)

after_fold = fold.reflect_direction(beam.chief_ray.d)
after_fold = after_fold / np.linalg.norm(after_fold)

mask.point = fold.point + 25e-3 * after_fold
mask.normal = after_fold
mask.__post_init__()

# Reflective DM sends the post-fold beam toward +y.
dm_out = np.array([0.0, 1.0, 0.0])
dm_out = dm_out / np.linalg.norm(dm_out)

dm_normal2 = after_fold - dm_out
dm_normal2 = dm_normal2 / np.linalg.norm(dm_normal2)

dm.point = fold.point + 55e-3 * after_fold
dm.normal = dm_normal2
dm.__post_init__()
dm.reflect_rays = True
dm.use_surface_normals = False
dm.set_commands(patterns["astigmatism-like"])

sh.point = dm.point + 35e-3 * dm_out
sh.normal = dm_out
sh.__post_init__()

bench = bt.OpticalBench3D()
for elem in [screen, fold, mask, dm, sh]:
    bench.add(elem)

fig = bench.plot_3d(
    beams=[beam],
    t=t_demo,
    s_end=0.04,
    title="Tutorial folded bench: screen → mirror → mask → DM → SH-WFS",
    max_rays_per_beam=80,
    ray_lw=0.9,
    ray_alpha=0.75,
    draw_elements=True,
    draw_labels=True,
    draw_normals=True,
    draw_local_axes=False,
    equal_axes=True,
    orthographic=True,
    view=(24, -55),
)
plt.show()

paths, traced = bench.trace_beam(beam, t=t_demo, s_end=0.04)
chief = traced[0]
print("Chief final position:", chief.r)
print("Chief final direction:", chief.d)
print("Chief OPD [nm]:", chief.opd * 1e9)
print("Chief field amplitude:", getattr(chief, "amplitude", None))

## Notes

- `beam_trace.py` is still a geometric ray-tracing layer. It does not perform Fresnel propagation between all optical planes.
- OPD-producing elements expose `sample_uv()`, which is the key handoff into `Wavefront2D` and the Fresnel modules.
- Amplitude-producing elements expose `sample_amplitude_uv()`.
- The DM can be used as a reflective ray element for folded geometry, but the safest AO simulation path remains to use its sampled OPD map in the wave-optics layer.